# since it is only static data we will use overwrite - no need to use upsert or scd2 here

In [0]:


from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType

# 1. SCHEMAS
bronze_schema = "7trents_data_platform.bronze"
silver_schema = "7trents_data_platform.silver"

# Ensure Silver database/schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Reading from: {bronze_schema}")
print(f"Writing to:   {silver_schema}")




In [0]:

# Helper: Trim whitespace on all string columns
def trim_strings(df):
    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))
    return df

In [0]:
# Quick Schema Check: Print exact Bronze column names
dimension_tables = [
    "dim_people", 
    "dim_county", 
    "dim_cancellation_reason", 
    "dim_shrinkage_type"
]

for tbl in dimension_tables:
    print(f"=== Columns in 7trents_data_platform.bronze.{tbl} ===")
    spark.table(f"7trents_data_platform.bronze.{tbl}").printSchema()

In [0]:
# ==============================================================================
# 2. PROCESS DIM_PEOPLE
# ==============================================================================
print("\n--- Processing: dim_people ---")
df_bronze_people = spark.table(f"{bronze_schema}.dim_people")

df_silver_people = (
    trim_strings(df_bronze_people)
    .filter(F.col("people_id").isNotNull())
    .dropDuplicates(["people_id"])
    .withColumn("people_id", F.col("people_id").cast(IntegerType()))
    .withColumn("hire_date", F.to_date(F.col("hire_date")))
    .withColumn("_silver_processed_at", F.current_timestamp())
    .select(
        "people_id",
        "first_name",
        "surname",
        "employee_name",
        "job_role",
        "employment_type",
        "team",
        "hire_date",
        "source_file",
        "load_date",
        "load_type",
        "_silver_processed_at"
    )
)

(
    df_silver_people.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.dim_people")
)
print(f"✓ Saved {silver_schema}.dim_people: {df_silver_people.count()} rows")

In [0]:
# ==============================================================================
# 3. PROCESS DIM_COUNTY
# ==============================================================================
print("\n--- Processing: dim_county ---")
df_bronze_county = spark.table(f"{bronze_schema}.dim_county")

df_silver_county = (
    trim_strings(df_bronze_county)
    .filter(F.col("county_id").isNotNull())
    .dropDuplicates(["county_id"])
    .withColumn("county_id", F.col("county_id").cast(IntegerType()))
    .withColumn("_silver_processed_at", F.current_timestamp())
    .select(
        "county_id",
        "county_name",
        "region",
        "country",
        "population_band",
        "urban_rural",
        "source_file",
        "load_date",
        "load_type",
        "_silver_processed_at"
    )
)

(
    df_silver_county.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.dim_county")
)
print(f"✓ Saved {silver_schema}.dim_county: {df_silver_county.count()} rows")

In [0]:


# ==============================================================================
# 4. PROCESS DIM_CANCELLATION_REASON
# ==============================================================================
print("\n--- Processing: dim_cancellation_reason ---")
df_bronze_cancel = spark.table(f"{bronze_schema}.dim_cancellation_reason")

df_silver_cancel = (
    trim_strings(df_bronze_cancel)
    .filter(F.col("cancellation_id").isNotNull())
    .dropDuplicates(["cancellation_id"])
    .withColumn("cancellation_id", F.col("cancellation_id").cast(IntegerType()))
    .withColumn("_silver_processed_at", F.current_timestamp())
    .select(
        "cancellation_id",
        "cancellation_reason",
        "reason_group",
        "is_customer_driven",
        "severity",
        "source_file",
        "load_date",
        "load_type",
        "_silver_processed_at"
    )
)

(
    df_silver_cancel.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.dim_cancellation_reason")
)
print(f"✓ Saved {silver_schema}.dim_cancellation_reason: {df_silver_cancel.count()} rows")

In [0]:

# ==============================================================================
# 5. PROCESS DIM_SHRINKAGE_TYPE
# ==============================================================================
print("\n--- Processing: dim_shrinkage_type ---")
df_bronze_shrink = spark.table(f"{bronze_schema}.dim_shrinkage_type")

df_silver_shrink = (
    trim_strings(df_bronze_shrink)
    .filter(F.col("shrinkage_type_id").isNotNull())
    .dropDuplicates(["shrinkage_type_id"])
    .withColumn("shrinkage_type_id", F.col("shrinkage_type_id").cast(IntegerType()))
    .withColumn("_silver_processed_at", F.current_timestamp())
    .select(
        "shrinkage_type_id",
        "shrinkage_type",
        "category",
        "is_paid",
        "is_planned",
        "source_file",
        "load_date",
        "load_type",
        "_silver_processed_at"
    )
)

(
    df_silver_shrink.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.dim_shrinkage_type")
)
print(f"✓ Saved {silver_schema}.dim_shrinkage_type: {df_silver_shrink.count()} rows")

In [0]:

# ==============================================================================
# 6. VERIFICATION AUDIT
# ==============================================================================
print("\n" + "="*50)
print("SILVER DIMENSION SUMMARY:")
print("="*50)

tables = ["dim_people", "dim_county", "dim_cancellation_reason", "dim_shrinkage_type"]
for t in tables:
    total = spark.table(f"{silver_schema}.{t}").count()
    print(f"Table: {silver_schema}.{t:<26} Rows: {total}")